# 🏊 Design Pattern: Multi-Agent Workflow Parallelization via Worker Pool & Semaphore

## Pattern Overview

This notebook illustrates the **Worker Pool with Semaphore Pattern** for parallelizing multi-agent workflows. This is a skeleton implementation demonstrating the pattern structure.

## Pattern Intent

Enable controlled parallel execution of tasks by using a fixed-size thread pool with semaphore-based concurrency limiting.

## Pattern Structure

```
┌─────────────────────────────────────────────────────────────────────────────┐
│              WORKER POOL + SEMAPHORE PARALLELIZATION PATTERN                │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│   ┌─────────────────┐                                                       │
│   │    PRODUCER     │                                                       │
│   │  (Orchestrator) │                                                       │
│   │                 │                                                       │
│   │  • Create tasks │                                                       │
│   │  • Submit ALL   │                                                       │
│   │    at once      │                                                       │
│   └────────┬────────┘                                                       │
│            │ submit all tasks                                               │
│            ▼                                                                │
│   ┌─────────────────────────────────────────────────────────────────────┐   │
│   │                         WORKER POOL                                  │   │
│   │  ┌─────────────────────────────────────────────────────────────────┐│   │
│   │  │              🚦 SEMAPHORE (max_workers=N)                        ││   │
│   │  │                                                                  ││   │
│   │  │   acquire() ─────┬─────────────┬─────────────┬───── ...         ││   │
│   │  │                  ▼             ▼             ▼                  ││   │
│   │  │            ┌─────────┐   ┌─────────┐   ┌─────────┐              ││   │
│   │  │            │ Slot 1  │   │ Slot 2  │   │ Slot N  │              ││   │
│   │  │            │(active) │   │(active) │   │(waiting)│              ││   │
│   │  │            └────┬────┘   └────┬────┘   └────┬────┘              ││   │
│   │  │                 │             │             │                   ││   │
│   │  │   release() ◄───┴─────────────┴─────────────┴───── ...         ││   │
│   │  └─────────────────────────────────────────────────────────────────┘│   │
│   │                                                                      │   │
│   │  ┌─────────────────────────────────────────────────────────────────┐│   │
│   │  │                 ThreadPoolExecutor                               ││   │
│   │  │  • Manages thread lifecycle                                      ││   │
│   │  │  • Reuses threads (no creation overhead)                         ││   │
│   │  │  • Returns Future objects                                        ││   │
│   │  └─────────────────────────────────────────────────────────────────┘│   │
│   └──────────────────────────────┬──────────────────────────────────────┘   │
│                                  │ all futures complete                     │
│                                  ▼                                          │
│                        ┌─────────────────┐                                  │
│                        │   AGGREGATOR    │                                  │
│                        │  • Collect      │                                  │
│                        │  • Synthesize   │                                  │
│                        └─────────────────┘                                  │
└─────────────────────────────────────────────────────────────────────────────┘
```

## Key Participants

| Participant | Role |
|-------------|------|
| **Task** | Unit of work to be processed |
| **Semaphore** | Controls max concurrent executions (counting lock) |
| **WorkerPool** | Manages threads, semaphore, and task execution |
| **ThreadPoolExecutor** | Handles thread lifecycle and reuse |
| **Producer (Orchestrator)** | Creates all tasks upfront |
| **Aggregator** | Collects and synthesizes results |

## When to Use

- All tasks are known upfront (batch processing)
- Need true parallel execution (not round-robin)
- Want to limit concurrent resource usage (API rate limits)
- Prefer thread reuse over thread creation per task

## Task Queue vs Worker Pool Comparison

| Aspect | Task Queue | Worker Pool + Semaphore |
|--------|------------|------------------------|
| Task Submission | Push one by one | Submit all at once |
| Execution Model | Round-robin | True parallel |
| Concurrency Control | Queue emptiness | Semaphore count |
| Blocking | Non-blocking pop | Blocking acquire |
| Best For | Streaming tasks | Batch processing |

---

## Part 1: Core Data Structures

### 1.1 Task Status Enumeration

The task lifecycle states.

In [ ]:
from enum import Enum
from typing import Any, Optional, List, Dict, Callable
from dataclasses import dataclass, field
from threading import Semaphore, Lock
from concurrent.futures import ThreadPoolExecutor, Future
from abc import ABC, abstractmethod


class TaskStatus(Enum):
    """
    Enumeration of possible task states in the lifecycle.
    
    State Transitions:
        PENDING → IN_PROGRESS → COMPLETED
                             → FAILED
    """
    PENDING = "pending"          # Task created, waiting for slot
    IN_PROGRESS = "in_progress"  # Task acquired semaphore slot, executing
    COMPLETED = "completed"      # Task finished successfully
    FAILED = "failed"            # Task encountered an error

### 1.2 Task Data Class

Represents a unit of work to be processed.

In [ ]:
@dataclass
class Task:
    """
    Represents a single unit of work in the worker pool pattern.
    
    This is the data object that:
    - Gets created by the Producer
    - Gets submitted to the WorkerPool
    - Gets executed when semaphore slot is acquired
    
    Attributes:
        task_id: Unique identifier for tracking
        payload: The actual work data (query, parameters, etc.)
        status: Current state in the lifecycle
        assigned_worker: ID of worker executing this task
        result: Output after processing (None until completed)
        error: Error message if failed (None if successful)
        metadata: Optional additional tracking information
    
    Usage:
        task = Task(task_id="001", payload={"query": "search term"})
        pool.register_task(task)
    """
    task_id: str
    payload: Any                                    # The work to be done
    status: TaskStatus = TaskStatus.PENDING
    assigned_worker: Optional[str] = None           # Set when executing
    result: Optional[Any] = None                    # Filled by worker
    error: Optional[str] = None                     # Filled on failure
    metadata: Dict[str, Any] = field(default_factory=dict)
    
    def mark_in_progress(self, worker_id: str) -> None:
        """Mark task as being processed by a worker."""
        self.status = TaskStatus.IN_PROGRESS
        self.assigned_worker = worker_id
    
    def mark_completed(self, result: Any) -> None:
        """Mark task as successfully completed with result."""
        self.status = TaskStatus.COMPLETED
        self.result = result
    
    def mark_failed(self, error: str) -> None:
        """Mark task as failed with error message."""
        self.status = TaskStatus.FAILED
        self.error = error

---

## Part 2: The Worker Pool

The central component that manages semaphore-controlled parallel execution.

In [ ]:
class WorkerPool:
    """
    Manages a pool of workers with semaphore-controlled concurrency.
    
    This is the central coordination component:
    - Semaphore limits concurrent task execution
    - ThreadPoolExecutor manages thread lifecycle
    - Tasks are submitted and executed in parallel
    
    Key Properties:
    - True parallelism: Tasks run concurrently (up to max_workers)
    - Automatic blocking: New tasks wait when all slots are busy
    - Thread reuse: Executor manages thread pool efficiently
    
    Semaphore Behavior:
    ┌─────────────────────────────────────────────────────────────────┐
    │  Semaphore (max_workers=3)                                      │
    │                                                                 │
    │  Counter: 3 → 2 → 1 → 0 → [BLOCKS] → 1 → 0 → ...               │
    │            ↑    ↑    ↑       ↑        ↑                        │
    │         acquire acquire acquire    release                      │
    │         (Task1) (Task2) (Task3)    (Task1)                      │
    └─────────────────────────────────────────────────────────────────┘
    """
    
    def __init__(self, max_workers: int = 3):
        """
        Initialize the worker pool.
        
        Args:
            max_workers: Maximum concurrent task executions
        """
        self.max_workers = max_workers
        
        # Concurrency primitives
        self._semaphore = Semaphore(max_workers)   # Controls concurrency
        self._lock = Lock()                         # Protects shared state
        
        # Thread pool for execution
        self._executor = ThreadPoolExecutor(max_workers=max_workers)
        
        # State tracking
        self._tasks: Dict[str, Task] = {}          # All registered tasks
        self._results: List[Dict[str, Any]] = []   # Completed results
        self._active_workers = 0                    # Currently executing
    
    # ─────────────────────────────────────────────────────────────────────
    # SEMAPHORE OPERATIONS
    # ─────────────────────────────────────────────────────────────────────
    
    def acquire_slot(self) -> None:
        """
        Acquire a semaphore slot (blocks if none available).
        
        This is called BEFORE task execution begins.
        Blocks the calling thread until a slot is available.
        """
        self._semaphore.acquire()
        with self._lock:
            self._active_workers += 1
    
    def release_slot(self) -> None:
        """
        Release a semaphore slot (wakes waiting threads).
        
        This is called AFTER task execution completes (in finally block).
        Allows another waiting task to proceed.
        """
        with self._lock:
            self._active_workers -= 1
        self._semaphore.release()
    
    # ─────────────────────────────────────────────────────────────────────
    # TASK MANAGEMENT
    # ─────────────────────────────────────────────────────────────────────
    
    def register_task(self, task: Task) -> None:
        """
        Register a task with the pool.
        
        Args:
            task: The task to register
        """
        with self._lock:
            self._tasks[task.task_id] = task
    
    def submit_task(self, task: Task, worker_id: str, 
                    execute_fn: Callable[[Any], Any]) -> Future:
        """
        Submit a task for execution in the thread pool.
        
        Args:
            task: The task to execute
            worker_id: ID of the worker handling this task
            execute_fn: Function that executes the task payload
            
        Returns:
            Future object for tracking completion
        """
        return self._executor.submit(
            self._execute_with_semaphore,
            task, worker_id, execute_fn
        )
    
    def _execute_with_semaphore(self, task: Task, worker_id: str,
                                 execute_fn: Callable[[Any], Any]) -> Dict:
        """
        Execute a task with semaphore protection.
        
        Pattern:
            1. acquire() - Wait for slot
            2. try: Execute task
            3. finally: release() - Always release slot
        
        Returns:
            Result dict with status and output
        """
        self.acquire_slot()
        
        try:
            task.mark_in_progress(worker_id)
            
            # Execute the actual work
            result = execute_fn(task.payload)
            
            # Record success
            task.mark_completed(result)
            self.add_result(task.task_id, worker_id, result)
            
            return {
                "task_id": task.task_id,
                "worker_id": worker_id,
                "status": "completed",
                "result": result
            }
            
        except Exception as e:
            # Record failure
            task.mark_failed(str(e))
            self.mark_failed(task.task_id, str(e))
            
            return {
                "task_id": task.task_id,
                "worker_id": worker_id,
                "status": "failed",
                "error": str(e)
            }
            
        finally:
            # CRITICAL: Always release the slot
            self.release_slot()
    
    # ─────────────────────────────────────────────────────────────────────
    # RESULT TRACKING
    # ─────────────────────────────────────────────────────────────────────
    
    def add_result(self, task_id: str, worker_id: str, result: Any) -> None:
        """Record a successful task result."""
        with self._lock:
            self._results.append({
                "task_id": task_id,
                "worker_id": worker_id,
                "result": result,
                "status": "completed"
            })
    
    def mark_failed(self, task_id: str, error: str) -> None:
        """Record a task failure."""
        with self._lock:
            self._results.append({
                "task_id": task_id,
                "error": error,
                "status": "failed"
            })
    
    def get_all_results(self) -> List[Dict]:
        """Get all completed results for aggregation."""
        with self._lock:
            return self._results.copy()
    
    # ─────────────────────────────────────────────────────────────────────
    # STATUS & CLEANUP
    # ─────────────────────────────────────────────────────────────────────
    
    def get_active_count(self) -> int:
        """Get number of currently executing tasks."""
        with self._lock:
            return self._active_workers
    
    def get_available_slots(self) -> int:
        """Get number of available semaphore slots."""
        with self._lock:
            return self.max_workers - self._active_workers
    
    def get_stats(self) -> Dict[str, int]:
        """Get pool statistics."""
        with self._lock:
            return {
                "max_workers": self.max_workers,
                "active": self._active_workers,
                "available": self.max_workers - self._active_workers,
                "total_tasks": len(self._tasks),
                "completed": sum(1 for t in self._tasks.values() 
                               if t.status == TaskStatus.COMPLETED),
                "failed": sum(1 for t in self._tasks.values() 
                            if t.status == TaskStatus.FAILED),
            }
    
    def shutdown(self) -> None:
        """Shutdown the thread pool."""
        self._executor.shutdown(wait=True)

## Part 3: Producer (Orchestrator)

The component that creates all tasks upfront and registers them with the pool.

In [ ]:
class Producer(ABC):
    """
    Abstract base class for task producers (Orchestrators).
    
    In the Worker Pool pattern, the Producer:
    1. Analyzes input (e.g., user query)
    2. Decomposes work into independent tasks
    3. Registers ALL tasks with the pool upfront
    
    Key Difference from Task Queue Pattern:
        Task Queue: Push tasks one-by-one to queue
        Worker Pool: Register all tasks, then submit all at once
    
    Subclass this to implement domain-specific task creation logic.
    """
    
    def __init__(self, pool: WorkerPool):
        """
        Initialize producer with reference to worker pool.
        
        Args:
            pool: The WorkerPool to register tasks with
        """
        self.pool = pool
        self._task_counter = 0
    
    @abstractmethod
    def decompose(self, input_data: Any) -> List[Dict]:
        """
        Decompose input into task payloads.
        
        Args:
            input_data: The input to break down (e.g., user query)
            
        Returns:
            List of task payload dictionaries
            
        IMPLEMENT THIS: Define how to split work into tasks.
        """
        pass
    
    def produce(self, input_data: Any) -> List[Task]:
        """
        Main producer method: decompose input and register all tasks.
        
        Args:
            input_data: The input to process
            
        Returns:
            List of created Task objects (for submission to pool)
        """
        payloads = self.decompose(input_data)
        tasks = []
        
        for payload in payloads:
            self._task_counter += 1
            task = Task(
                task_id=f"task_{self._task_counter:04d}",
                payload=payload
            )
            self.pool.register_task(task)
            tasks.append(task)
        
        return tasks


# ─────────────────────────────────────────────────────────────────────────────
# EXAMPLE: Concrete Producer Implementation
# ─────────────────────────────────────────────────────────────────────────────

class QueryDecomposer(Producer):
    """
    Example producer that decomposes a query into search tasks.
    
    This is a skeleton - in real implementation, you might use
    an LLM to intelligently decompose the query.
    """
    
    def decompose(self, input_data: Any) -> List[Dict]:
        """
        Decompose a user query into search sub-tasks.
        
        Args:
            input_data: User query string or dict with 'query' key
            
        Returns:
            List of search task payloads
        """
        query = input_data if isinstance(input_data, str) else input_data.get("query", "")
        
        # SKELETON: Replace with actual decomposition logic
        # In real implementation, use LLM to create sub-queries
        sub_queries = [
            f"{query} - overview",
            f"{query} - recent developments",
            f"{query} - best practices",
            f"{query} - future trends",
        ]
        
        return [{"search_query": sq} for sq in sub_queries]

## Part 4: Task Executor

The component that defines how individual tasks are executed.

In [ ]:
class TaskExecutor(ABC):
    """
    Abstract base class for task executors.
    
    The TaskExecutor defines HOW to process a task payload.
    Unlike the Task Queue pattern's Consumer, the executor
    doesn't pull tasks - it's called by the pool.
    
    Pattern Role:
        Pool submits task → Executor processes payload → Result returned
    
    Subclass this to implement domain-specific execution logic.
    """
    
    @abstractmethod
    def execute(self, payload: Any) -> Any:
        """
        Execute the task work.
        
        Args:
            payload: The task payload containing work data
            
        Returns:
            The result of the task execution
            
        Raises:
            Exception: If task execution fails
            
        IMPLEMENT THIS: Define how to process a task.
        """
        pass


# ─────────────────────────────────────────────────────────────────────────────
# EXAMPLE: Concrete TaskExecutor Implementation
# ─────────────────────────────────────────────────────────────────────────────

class SearchExecutor(TaskExecutor):
    """
    Example executor that performs search operations.
    
    This is a skeleton - in real implementation, you would
    call an actual search API (e.g., Tavily, Google, etc.).
    """
    
    def execute(self, payload: Any) -> Any:
        """
        Execute a search task.
        
        Args:
            payload: Dict with 'search_query' key
            
        Returns:
            Search results (simulated in skeleton)
        """
        import time
        query = payload.get("search_query", "")
        
        # SKELETON: Replace with actual search implementation
        # Simulate some work
        time.sleep(0.1)
        
        return {
            "query": query,
            "results": [f"Result 1 for: {query}", f"Result 2 for: {query}"],
            "source": "skeleton_search"
        }

## Part 5: Aggregator

The component that collects and synthesizes results from all workers.

In [ ]:
class Aggregator(ABC):
    """
    Abstract base class for result aggregators.
    
    The Aggregator collects results after all futures complete
    and synthesizes them into a final output.
    
    Pattern Role:
        All tasks complete → Aggregator collects → Synthesizes final output
    
    Subclass this to implement domain-specific synthesis logic.
    """
    
    def __init__(self, pool: WorkerPool):
        """
        Initialize aggregator with reference to worker pool.
        
        Args:
            pool: The WorkerPool to collect results from
        """
        self.pool = pool
    
    @abstractmethod
    def synthesize(self, results: List[Dict]) -> Any:
        """
        Synthesize multiple results into a final output.
        
        Args:
            results: List of result dicts from workers
            
        Returns:
            Synthesized final output
            
        IMPLEMENT THIS: Define how to combine results.
        """
        pass
    
    def aggregate(self) -> Any:
        """
        Main aggregation method: collect results and synthesize.
        
        Returns:
            Final synthesized output
        """
        results = self.pool.get_all_results()
        successful_results = [r for r in results if r.get("status") == "completed"]
        return self.synthesize(successful_results)


# ─────────────────────────────────────────────────────────────────────────────
# EXAMPLE: Concrete Aggregator Implementation
# ─────────────────────────────────────────────────────────────────────────────

class ResultSynthesizer(Aggregator):
    """
    Example aggregator that combines search results.
    
    This is a skeleton - in real implementation, you might use
    an LLM to intelligently synthesize the results.
    """
    
    def synthesize(self, results: List[Dict]) -> Any:
        """
        Synthesize search results into a report.
        
        Args:
            results: List of search result dicts
            
        Returns:
            Combined report dict
        """
        # SKELETON: Replace with actual synthesis logic
        all_findings = []
        for r in results:
            if "result" in r and "results" in r["result"]:
                all_findings.extend(r["result"]["results"])
        
        return {
            "summary": "Combined results from parallel search execution",
            "total_results": len(all_findings),
            "findings": all_findings,
            "stats": self.pool.get_stats()
        }

## Part 6: Workflow Coordinator

The component that orchestrates the entire pattern execution.

In [ ]:
class WorkflowCoordinator:
    """
    Coordinates the worker pool parallelization workflow.
    
    This is the "glue" that ties the pattern together:
    1. Creates the worker pool
    2. Runs the producer to create all tasks
    3. Submits ALL tasks to the pool at once
    4. Waits for all futures to complete
    5. Runs the aggregator to synthesize results
    
    Key Difference from Task Queue Pattern:
        Task Queue: Process in rounds until queue empty
        Worker Pool: Submit all, wait for all futures
    
    Workflow:
        ┌──────────────┐     ┌───────────────────────────────────┐     ┌────────────┐
        │   Producer   │ ──► │         Worker Pool               │ ──► │ Aggregator │
        │  (create all)│     │  submit() → futures → results     │     │ (synthesize)│
        └──────────────┘     └───────────────────────────────────┘     └────────────┘
    """
    
    def __init__(
        self,
        producer: Producer,
        executor: TaskExecutor,
        aggregator: Aggregator,
        pool: WorkerPool
    ):
        """
        Initialize the workflow coordinator.
        
        Args:
            producer: The Producer instance (orchestrator)
            executor: The TaskExecutor for processing payloads
            aggregator: The Aggregator instance
            pool: The WorkerPool for parallel execution
        """
        self.producer = producer
        self.executor = executor
        self.aggregator = aggregator
        self.pool = pool
    
    def run(self, input_data: Any) -> Any:
        """
        Execute the complete workflow.
        
        Args:
            input_data: Input to the producer (e.g., user query)
            
        Returns:
            Final aggregated result
            
        Workflow Steps:
            1. Producer creates all tasks
            2. All tasks submitted to pool (true parallel)
            3. Wait for all futures to complete
            4. Aggregator synthesizes results
        """
        # ─────────────────────────────────────────────────────────────────────
        # PHASE 1: Production - Create all tasks upfront
        # ─────────────────────────────────────────────────────────────────────
        print(f"📋 PRODUCER: Creating tasks...")
        tasks = self.producer.produce(input_data)
        print(f"   Created {len(tasks)} tasks")
        
        # ─────────────────────────────────────────────────────────────────────
        # PHASE 2: Submission - Submit ALL tasks to pool at once
        # ─────────────────────────────────────────────────────────────────────
        print(f"\n🏊 POOL: Submitting all tasks (max_workers={self.pool.max_workers})...")
        
        futures = []
        for i, task in enumerate(tasks):
            worker_id = f"worker_{(i % self.pool.max_workers) + 1}"
            future = self.pool.submit_task(task, worker_id, self.executor.execute)
            futures.append(future)
        
        print(f"   Submitted {len(futures)} tasks")
        
        # ─────────────────────────────────────────────────────────────────────
        # PHASE 3: Wait - Block until all futures complete
        # ─────────────────────────────────────────────────────────────────────
        print(f"\n⏳ WAITING: For all tasks to complete...")
        
        results = []
        for i, future in enumerate(futures):
            result = future.result()  # Blocks until complete
            status = "✅" if result["status"] == "completed" else "❌"
            print(f"   {status} Task {i+1}/{len(futures)}: {result['task_id']}")
            results.append(result)
        
        print(f"\n   Pool stats: {self.pool.get_stats()}")
        
        # ─────────────────────────────────────────────────────────────────────
        # PHASE 4: Aggregation - Synthesize results
        # ─────────────────────────────────────────────────────────────────────
        print(f"\n📊 AGGREGATOR: Synthesizing results...")
        final_result = self.aggregator.aggregate()
        print(f"   Done!")
        
        # Cleanup
        self.pool.shutdown()
        
        return final_result

---

## Part 7: Usage Example

Putting all the components together.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# USAGE EXAMPLE: Running the Worker Pool + Semaphore Pattern
# ═══════════════════════════════════════════════════════════════════════════════

def run_example(max_workers: int = 2):
    """
    Demonstrates the complete worker pool parallelization pattern.
    
    Args:
        max_workers: Maximum concurrent task executions
    """
    print("=" * 70)
    print("WORKER POOL + SEMAPHORE PARALLELIZATION PATTERN - DEMO")
    print(f"Max Workers (Semaphore Slots): {max_workers}")
    print("=" * 70)
    
    # Step 1: Create the worker pool
    pool = WorkerPool(max_workers=max_workers)
    
    # Step 2: Create pattern components
    producer = QueryDecomposer(pool)
    executor = SearchExecutor()
    aggregator = ResultSynthesizer(pool)
    
    # Step 3: Create coordinator
    coordinator = WorkflowCoordinator(
        producer=producer,
        executor=executor,
        aggregator=aggregator,
        pool=pool
    )
    
    # Step 4: Run the workflow
    result = coordinator.run("machine learning applications")
    
    # Step 5: Display result
    print("\n" + "=" * 70)
    print("FINAL RESULT")
    print("=" * 70)
    print(f"Summary: {result['summary']}")
    print(f"Total findings: {result['total_results']}")
    print(f"Stats: {result['stats']}")
    print("\nFindings:")
    for i, finding in enumerate(result['findings'], 1):
        print(f"  {i}. {finding}")
    
    return result


# Run the example with 2 concurrent workers
example_result = run_example(max_workers=2)

---

## Part 8: Pattern Summary

### Class Diagram

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                              CLASS DIAGRAM                                  │
└─────────────────────────────────────────────────────────────────────────────┘

    ┌─────────────────┐           ┌─────────────────────────────────────┐
    │   <<enum>>      │           │              Task                   │
    │   TaskStatus    │           │─────────────────────────────────────│
    │─────────────────│           │ + task_id: str                      │
    │ PENDING         │◄──────────│ + payload: Any                      │
    │ IN_PROGRESS     │           │ + status: TaskStatus                │
    │ COMPLETED       │           │ + assigned_worker: Optional[str]    │
    │ FAILED          │           │ + result: Optional[Any]             │
    └─────────────────┘           │ + error: Optional[str]              │
                                  │─────────────────────────────────────│
                                  │ + mark_in_progress(worker_id)       │
                                  │ + mark_completed(result)            │
                                  │ + mark_failed(error)                │
                                  └───────────────┬─────────────────────┘
                                                  │ manages
                                                  ▼
┌────────────────────────────────────────────────────────────────────────────┐
│                               WorkerPool                                    │
│────────────────────────────────────────────────────────────────────────────│
│ + max_workers: int                                                         │
│ - _semaphore: Semaphore          ◄─── Controls concurrent access           │
│ - _lock: Lock                                                              │
│ - _executor: ThreadPoolExecutor  ◄─── Manages thread lifecycle             │
│ - _tasks: Dict[str, Task]                                                  │
│ - _results: List[Dict]                                                     │
│ - _active_workers: int                                                     │
│────────────────────────────────────────────────────────────────────────────│
│ + acquire_slot(): None           # Semaphore acquire (blocks)              │
│ + release_slot(): None           # Semaphore release (wakes)               │
│ + register_task(task): None                                                │
│ + submit_task(task, worker_id, execute_fn): Future                         │
│ - _execute_with_semaphore(task, worker_id, fn): Dict                       │
│ + add_result(task_id, worker_id, result): None                             │
│ + mark_failed(task_id, error): None                                        │
│ + get_all_results(): List[Dict]                                            │
│ + get_stats(): Dict[str, int]                                              │
│ + shutdown(): None                                                         │
└─────────────────────────────────────────────────────────────────────────────┘
         ▲                                                     ▲
         │ uses                                                │ uses
         │                                                     │
┌────────┴────────┐        ┌─────────────────┐        ┌────────┴────────┐
│   <<abstract>>  │        │   <<abstract>>  │        │   <<abstract>>  │
│    Producer     │        │  TaskExecutor   │        │   Aggregator    │
│─────────────────│        │─────────────────│        │─────────────────│
│ + pool          │        │                 │        │ + pool          │
│─────────────────│        │─────────────────│        │─────────────────│
│ + decompose()*  │        │ + execute()*    │        │ + synthesize()* │
│ + produce()     │        │                 │        │ + aggregate()   │
└────────┬────────┘        └────────┬────────┘        └────────┬────────┘
         │ extends                  │ extends                  │ extends
         ▼                          ▼                          ▼
┌─────────────────┐        ┌─────────────────┐        ┌─────────────────┐
│ QueryDecomposer │        │ SearchExecutor  │        │ResultSynthesizer│
│─────────────────│        │─────────────────│        │─────────────────│
│ + decompose()   │        │ + execute()     │        │ + synthesize()  │
└─────────────────┘        └─────────────────┘        └─────────────────┘

                    ┌─────────────────────────────────────────────┐
                    │          WorkflowCoordinator                │
                    │─────────────────────────────────────────────│
                    │ + producer: Producer                        │
                    │ + executor: TaskExecutor                    │
                    │ + aggregator: Aggregator                    │
                    │ + pool: WorkerPool                          │
                    │─────────────────────────────────────────────│
                    │ + run(input_data): Any                      │
                    └─────────────────────────────────────────────┘
```

### Sequence Diagram

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                            SEQUENCE DIAGRAM                                 │
│                     (Worker Pool + Semaphore Pattern)                       │
└─────────────────────────────────────────────────────────────────────────────┘

  Coordinator     Producer        Pool          Semaphore     Executor    Aggregator
       │              │             │               │             │             │
       │  produce()   │             │               │             │             │
       │─────────────►│             │               │             │             │
       │              │ register(T1)│               │             │             │
       │              │────────────►│               │             │             │
       │              │ register(T2)│               │             │             │
       │              │────────────►│               │             │             │
       │              │ register(T3)│               │             │             │
       │              │────────────►│               │             │             │
       │◄─────────────│             │               │             │             │
       │   [tasks]    │             │               │             │             │
       │              │             │               │             │             │
       │──────────────┼─────────────┼───────────────┼─────────────┼─────────────│
       │              │ SUBMIT ALL TASKS AT ONCE    │             │             │
       │──────────────┼─────────────┼───────────────┼─────────────┼─────────────│
       │              │             │               │             │             │
       │    submit(T1, executor)    │               │             │             │
       │───────────────────────────►│   acquire()   │             │             │
       │              │             │──────────────►│             │             │
       │              │             │   ✓ (cnt=2)   │             │             │
       │              │             │◄──────────────│  execute()  │             │
       │              │             │──────────────────────────────────────────►│
       │    submit(T2, executor)    │               │             │             │
       │───────────────────────────►│   acquire()   │             │             │
       │              │             │──────────────►│             │             │
       │              │             │   ✓ (cnt=1)   │             │             │
       │              │             │◄──────────────│  execute()  │             │
       │              │             │──────────────────────────────────────────►│
       │    submit(T3, executor)    │               │             │             │
       │───────────────────────────►│   acquire()   │             │             │
       │              │             │──────────────►│             │             │
       │              │             │  BLOCKS(cnt=0)│             │             │
       │              │             │               │             │             │
       │──────────────┼─────────────┼───────────────┼─────────────┼─────────────│
       │              │ PARALLEL EXECUTION (up to max_workers)    │             │
       │──────────────┼─────────────┼───────────────┼─────────────┼─────────────│
       │              │             │               │  T1 done    │             │
       │              │             │◄──────────────────────────────────────────│
       │              │             │   release()   │             │             │
       │              │             │──────────────►│             │             │
       │              │             │   (cnt=1)     │  T3 wakes   │             │
       │              │             │◄──────────────│  execute()  │             │
       │              │             │──────────────────────────────────────────►│
       │              │             │               │  T2 done    │             │
       │              │             │◄──────────────────────────────────────────│
       │              │             │   release()   │             │             │
       │              │             │──────────────►│             │             │
       │              │             │               │  T3 done    │             │
       │              │             │◄──────────────────────────────────────────│
       │              │             │   release()   │             │             │
       │              │             │──────────────►│             │             │
       │              │             │               │             │             │
       │──────────────┼─────────────┼───────────────┼─────────────┼─────────────│
       │              │       ALL FUTURES COMPLETE  │             │             │
       │──────────────┼─────────────┼───────────────┼─────────────┼─────────────│
       │              │             │               │             │             │
       │  aggregate() │             │               │             │             │
       │──────────────┼─────────────┼───────────────┼─────────────┼────────────►│
       │              │             │ get_results() │             │             │
       │              │             │◄──────────────┼─────────────┼─────────────│
       │              │             │──────────────────────────────────────────►│
       │              │             │               │             │  synthesize │
       │◄─────────────────────────────────────────────────────────────────────│
       │  final_result│             │               │             │             │
```

### Key Takeaways

| Concept | Description |
|---------|-------------|
| **Semaphore** | Counting lock that allows N concurrent accesses (not just 1 like mutex) |
| **True Parallelism** | Tasks run concurrently, not in sequential rounds |
| **Blocking Acquire** | Tasks wait automatically when all slots are busy |
| **Thread Reuse** | ThreadPoolExecutor manages thread lifecycle efficiently |
| **Finally Block** | Critical for ensuring semaphore release on any exit path |
| **Batch Submission** | All tasks submitted at once, maximizing parallelism |

### Implementation Checklist

To implement this pattern for your use case:

1. ☐ Define your `Task` structure (what payload represents your work unit?)
2. ☐ Set `max_workers` based on resource constraints (API limits, CPU cores)
3. ☐ Implement `Producer.decompose()` (how to split input into tasks?)
4. ☐ Implement `TaskExecutor.execute()` (how to process one task?)
5. ☐ Implement `Aggregator.synthesize()` (how to combine results?)
6. ☐ Ensure `finally` block releases semaphore on all exit paths!
7. ☐ Call `pool.shutdown()` when done

### Semaphore vs Lock vs Queue

| Primitive | Concurrent Access | Blocking | Use Case |
|-----------|------------------|----------|----------|
| **Lock/Mutex** | 1 (exclusive) | Yes | Protect critical section |
| **Semaphore** | N (counting) | Yes | Limit concurrent resources |
| **Queue** | Many (FIFO) | Optional | Decouple producer/consumer |

### Related Patterns

- **Task Queue Pattern**: Push/pop tasks, round-robin execution
- **Thread Pool Pattern**: Pre-create threads, reuse for tasks
- **Bounded Buffer**: Producer-consumer with size limit
- **Rate Limiter**: Control request rate to external services

## References

- **Python Semaphore**: threading.Semaphore documentation
- **ThreadPoolExecutor**: concurrent.futures module documentation
- **Worker Pool Pattern**: Classic concurrency pattern for thread management
- **Full Implementation**: See `web_search_example_via_worker_pool_and_semaphore.ipynb` for a complete working example with LangGraph integration